# Lesson 23 — BLE 服务模型与 provisioning

用 `connected` 和 `text` 表示连接事件与特征写入：离线 advertising，连接后提供 Provisioning Service / WiFiCredentials Characteristic。它是协议模型，不声称模拟射频、发现、配对或加密。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 23-ble-provisioning guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (!device_.input.connected) {
                device_.output.state = "advertising";
                return;
            }
            if (device_.input.text.rfind("wifi:", 0) == 0 &&
                device_.input.text.size() > 5) {
                provisioned_ = true;
                device_.output.outgoing = "provisioning-status:accepted";
                ++device_.output.events;
            } else if (!device_.input.text.empty()) {
                device_.output.error = true;
                device_.output.outgoing = "provisioning-status:rejected";
            }
            device_.output.state = provisioned_ ? "provisioned" : "connected";
            device_.output.display =
                "Service: Provisioning / Characteristic: WiFiCredentials";
        }

    private:
        Device& device_;
        bool provisioned_ = false;
    };
}

## 2. Modify

把 `wifi:MakerNet` 改成 `wifi:`，观察空凭据是否被拒绝。

In [ ]:
%%academy_lab 23-ble-provisioning modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (!device_.input.connected) {
                device_.output.state = "advertising";
                return;
            }
            if (device_.input.text.rfind("wifi:", 0) == 0 &&
                device_.input.text.size() > 5) {
                provisioned_ = true;
                device_.output.outgoing = "provisioning-status:accepted";
                ++device_.output.events;
            } else if (!device_.input.text.empty()) {
                device_.output.error = true;
                device_.output.outgoing = "provisioning-status:rejected";
            }
            device_.output.state = provisioned_ ? "provisioned" : "connected";
            device_.output.display =
                "Service: Provisioning / Characteristic: WiFiCredentials";
        }

    private:
        Device& device_;
        bool provisioned_ = false;
    };
}

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 23-ble-provisioning broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (!device_.input.connected) {
                device_.output.state = "advertising";
                return;
            }
            if (device_.input.text.rfind("wifi:", 0) == 0) {
                device_.output.state = "provisioned";
                ++device_.output.events;
            }
        }

    private:
        Device& device_;
    };
}

## 4. Build

要求：未连接时 advertising；合法非空 Wi-Fi 写入得到 accepted；空凭据得到 rejected/error；显示服务和特征名称。

提示 1：先区分连接状态。提示 2：检查 `wifi:` 后是否有内容。提示 3：每次特征写入都要产生清晰 response。

In [ ]:
%%academy_lab 23-ble-provisioning exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() { /* TODO: 你的实现 */
        }

    private:
        Device& device_;
    };
}


## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？